[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaxiRuess/DeepLearning_101/blob/main/notebooks/08_JAX/03_XLA_vs_Triton.ipynb)

# XLA vs Triton — Compiler vs Hand-Written Kernels

The [Triton kernel notebooks](../06_Kernels/) taught us to write GPU kernels by hand. JAX's `jax.jit` compiles code via **XLA** (Accelerated Linear Algebra) automatically. This notebook asks: **when does the compiler beat the human, and when does it not?**

| | Triton (manual) | XLA via `jax.jit` (automatic) |
|---|---|---|
| Who writes the kernel? | You | The compiler |
| Optimization level | **Individual operation** (softmax, matmul) | **Entire computation graph** |
| Effort | High (pointer arithmetic, tiling, masking) | Low (`@jax.jit` decorator) |
| Control | Full | None (compiler decides) |
| Portability | NVIDIA GPU only | **GPU, TPU, CPU** |
| Fusion | Manual (you decide what to fuse) | **Automatic** (compiler finds fusion opportunities) |
| Used by | Meta, OpenAI | Google DeepMind |

We'll implement the same operations (softmax, matmul, layernorm, fused operations) in both frameworks and benchmark them.

## Setup

In [ ]:
import sys, os

if "google.colab" in str(get_ipython()):
    if not os.path.exists("/content/DeepLearning_101"):
        !git clone --depth 1 https://github.com/MaxiRuess/DeepLearning_101.git /content/DeepLearning_101
    os.chdir("/content/DeepLearning_101/notebooks/08_JAX")
    sys.path.insert(0, "/content/DeepLearning_101")
else:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

In [ ]:
try:
    import jax
    import jax.numpy as jnp
except ImportError:
    %pip install -q jax jaxlib
    import jax
    import jax.numpy as jnp

import torch
import numpy as np
import matplotlib.pyplot as plt
import time

HAS_CUDA = torch.cuda.is_available()
JAX_DEVICE = str(jax.devices()[0])

print(f"JAX version: {jax.__version__}, device: {JAX_DEVICE}")
print(f"PyTorch version: {torch.__version__}, CUDA: {HAS_CUDA}")

if HAS_CUDA:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## The Two Approaches to GPU Speed

```
Approach 1: Manual Kernels (Triton)
  You write: @triton.jit softmax_kernel(...)
  You control: grid, block size, pointer arithmetic, masking, tiling
  Result: one fused kernel that does exactly what you specified

Approach 2: Compiler Optimization (XLA via jax.jit)
  You write: @jax.jit def softmax(x): return jnp.exp(x - x.max()) / jnp.exp(x - x.max()).sum()
  Compiler does: trace → optimize → fuse → generate GPU code
  Result: one or more fused kernels the compiler decided on
```

The question isn't which is "better" — it's **when each approach wins**:
- XLA wins when standard ops compose well and the compiler can fuse them
- Triton wins when you need custom memory access patterns (like FlashAttention's online softmax)

## 1. Softmax — Where XLA Already Fuses

In the [Softmax kernel notebook](../06_Kernels/02_Softmax.ipynb), we showed that PyTorch's `torch.softmax` does 3 separate HBM passes. Our Triton kernel fused them into 1.

But what about `jax.jit`? Let's see if XLA can figure out the fusion on its own.

In [ ]:
# JAX: write the math, let XLA compile it
@jax.jit
def jax_softmax(x):
    """Softmax written as plain math — XLA fuses the ops."""
    x_max = jnp.max(x, axis=-1, keepdims=True)
    exp_x = jnp.exp(x - x_max)
    return exp_x / jnp.sum(exp_x, axis=-1, keepdims=True)

# Or just use the built-in
@jax.jit
def jax_softmax_builtin(x):
    return jax.nn.softmax(x, axis=-1)

# PyTorch naive (3 separate ops — what our Triton kernel beat)
def pytorch_softmax_naive(x):
    x_max = x.max(dim=-1, keepdim=True).values
    exp_x = torch.exp(x - x_max)
    return exp_x / exp_x.sum(dim=-1, keepdim=True)

# Verify correctness
x_np = np.random.randn(128, 512).astype(np.float32)
x_jax = jnp.array(x_np)

out1 = np.array(jax_softmax(x_jax))
out2 = np.array(jax_softmax_builtin(x_jax))

print(f"Manual vs builtin max diff: {np.abs(out1 - out2).max():.2e}")
print(f"Row sums: {out1.sum(axis=-1)[:3]}  (should be ~1.0)")

In [ ]:
# Look at what XLA actually compiles — the HLO (High Level Optimizer) graph
# This shows if XLA fused the ops or kept them separate
x_test = jnp.ones((4, 8))
lowered = jax.jit(jax_softmax).lower(x_test)
hlo = lowered.as_text()

# Count the number of "fusion" ops — more = better optimization
n_fusions = hlo.count("fusion")
n_ops = hlo.count("%")
print(f"XLA compiled softmax:")
print(f"  Fusion ops: {n_fusions}")
print(f"  Total ops:  {n_ops}")
print(f"\nXLA {'did' if n_fusions > 0 else 'did NOT'} fuse operations.")
print("\nFirst 500 chars of HLO:")
print(hlo[:500])

## 2. Matmul — Both Use Optimized Libraries

Matrix multiplication is special — both JAX and PyTorch call highly optimized libraries (cuBLAS on GPU, Eigen on CPU). Neither XLA nor Triton will significantly beat them for standard matmul.

Our [Matrix Multiply kernel](../06_Kernels/03_Matrix_Multiply.ipynb) showed that a hand-written Triton kernel *approaches* cuBLAS but doesn't beat it. Same story for XLA — it delegates to cuBLAS.

In [ ]:
@jax.jit
def jax_matmul(a, b):
    return a @ b

# Verify
a = jnp.array(np.random.randn(512, 512).astype(np.float32))
b = jnp.array(np.random.randn(512, 512).astype(np.float32))

result = jax_matmul(a, b)
ref = np.array(a) @ np.array(b)
print(f"Matmul max diff (JAX vs NumPy): {np.abs(np.array(result) - ref).max():.2e}")

## 3. LayerNorm — XLA's Sweet Spot

LayerNorm is where XLA shines. In the [LayerNorm kernel notebook](../06_Kernels/04_LayerNorm.ipynb), we fused 4 passes into 1. But XLA can figure this out automatically from the math:

In [ ]:
@jax.jit
def jax_layer_norm(x, gamma, beta, eps=1e-5):
    """LayerNorm written as plain math — XLA should fuse the 4 passes."""
    mean = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.var(x, axis=-1, keepdims=True)
    x_norm = (x - mean) / jnp.sqrt(var + eps)
    return gamma * x_norm + beta

@jax.jit
def jax_rms_norm(x, gamma, eps=1e-5):
    """RMSNorm — even simpler, only 1 reduction."""
    rms = jnp.sqrt(jnp.mean(x ** 2, axis=-1, keepdims=True) + eps)
    return gamma * (x / rms)

# Verify
x = jnp.array(np.random.randn(128, 512).astype(np.float32))
gamma = jnp.ones(512)
beta = jnp.zeros(512)

out_ln = jax_layer_norm(x, gamma, beta)
out_rms = jax_rms_norm(x, gamma)

# Check: normalized rows should have ~0 mean and ~1 variance
print(f"LayerNorm output — mean: {float(out_ln.mean()):.4f}, std: {float(out_ln.std()):.4f}")
print(f"RMSNorm output  — mean: {float(out_rms.mean()):.4f}, std: {float(out_rms.std()):.4f}")

## 4. Fused Operations — Where It Gets Interesting

XLA's real power is fusing **sequences of operations** that you'd never manually fuse. For example, a residual connection + layer norm + dropout:

In [ ]:
@jax.jit
def jax_residual_norm(x, residual, gamma, beta, eps=1e-5):
    """Residual + LayerNorm fused by XLA into minimal HBM passes."""
    x = x + residual
    mean = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.var(x, axis=-1, keepdims=True)
    x_norm = (x - mean) / jnp.sqrt(var + eps)
    return gamma * x_norm + beta

# In Triton, you'd write a custom kernel for this.
# In JAX, the compiler handles it automatically.
x = jnp.array(np.random.randn(128, 512).astype(np.float32))
residual = jnp.array(np.random.randn(128, 512).astype(np.float32))
gamma = jnp.ones(512)
beta = jnp.zeros(512)

out = jax_residual_norm(x, residual, gamma, beta)
print(f"Residual + LayerNorm output shape: {out.shape}")

# Check XLA fusion
lowered = jax.jit(jax_residual_norm).lower(x, residual, gamma, beta)
hlo = lowered.as_text()
n_fusions = hlo.count("fusion")
print(f"XLA fusions for residual+layernorm: {n_fusions}")

## 5. Benchmark: JAX (XLA) vs PyTorch

Since Triton only runs on NVIDIA GPUs and JAX/XLA runs on CPU too, we benchmark on **whatever device is available**. The comparison is between JAX's XLA compiler and PyTorch's eager execution.

In [ ]:
def benchmark_jax(fn, *args, warmup=10, iters=100):
    """Benchmark a JAX function (waits for async completion)."""
    for _ in range(warmup):
        _ = fn(*args)
    # Block until warmup is done
    jax.block_until_ready(fn(*args))

    start = time.perf_counter()
    for _ in range(iters):
        result = fn(*args)
    jax.block_until_ready(result)
    return (time.perf_counter() - start) / iters


def benchmark_pytorch(fn, *args, warmup=10, iters=100):
    """Benchmark a PyTorch function."""
    device = args[0].device if hasattr(args[0], 'device') else 'cpu'
    for _ in range(warmup):
        _ = fn(*args)
    if str(device) == 'cuda':
        torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(iters):
        _ = fn(*args)
    if str(device) == 'cuda':
        torch.cuda.synchronize()
    return (time.perf_counter() - start) / iters

In [ ]:
# Benchmark: Softmax, LayerNorm, RMSNorm, Residual+LayerNorm
sizes = [256, 512, 1024, 2048, 4096]
n_rows = 128

results = {
    'softmax': {'jax': [], 'pytorch': []},
    'layernorm': {'jax': [], 'pytorch': []},
    'rmsnorm': {'jax': [], 'pytorch': []},
    'residual_norm': {'jax': [], 'pytorch': []},
}

device = 'cuda' if HAS_CUDA else 'cpu'

for D in sizes:
    x_np = np.random.randn(n_rows, D).astype(np.float32)
    r_np = np.random.randn(n_rows, D).astype(np.float32)
    g_np = np.ones(D, dtype=np.float32)
    b_np = np.zeros(D, dtype=np.float32)

    x_jax, r_jax = jnp.array(x_np), jnp.array(r_np)
    g_jax, b_jax = jnp.array(g_np), jnp.array(b_np)

    x_pt = torch.tensor(x_np, device=device)
    r_pt = torch.tensor(r_np, device=device)
    g_pt = torch.tensor(g_np, device=device)
    b_pt = torch.tensor(b_np, device=device)

    # Softmax
    results['softmax']['jax'].append(benchmark_jax(jax_softmax, x_jax) * 1e6)
    results['softmax']['pytorch'].append(benchmark_pytorch(
        lambda x: torch.softmax(x, dim=-1), x_pt) * 1e6)

    # LayerNorm
    results['layernorm']['jax'].append(benchmark_jax(jax_layer_norm, x_jax, g_jax, b_jax) * 1e6)
    results['layernorm']['pytorch'].append(benchmark_pytorch(
        lambda x: torch.layer_norm(x, [D], g_pt, b_pt), x_pt) * 1e6)

    # RMSNorm
    results['rmsnorm']['jax'].append(benchmark_jax(jax_rms_norm, x_jax, g_jax) * 1e6)
    def pytorch_rmsnorm(x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + 1e-5)
        return g_pt * (x / rms)
    results['rmsnorm']['pytorch'].append(benchmark_pytorch(pytorch_rmsnorm, x_pt) * 1e6)

    # Residual + LayerNorm
    results['residual_norm']['jax'].append(benchmark_jax(
        jax_residual_norm, x_jax, r_jax, g_jax, b_jax) * 1e6)
    def pytorch_residual_norm(x):
        x = x + r_pt
        return torch.layer_norm(x, [D], g_pt, b_pt)
    results['residual_norm']['pytorch'].append(benchmark_pytorch(pytorch_residual_norm, x_pt) * 1e6)

    print(f"D={D:4d} done")

print(f"\nBenchmarked on: JAX={JAX_DEVICE}, PyTorch={device}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ops = ['softmax', 'layernorm', 'rmsnorm', 'residual_norm']
titles = ['Softmax', 'LayerNorm', 'RMSNorm', 'Residual + LayerNorm']

for ax, op, title in zip(axes.flat, ops, titles):
    ax.plot(sizes, results[op]['jax'], 'o-', label='JAX (XLA)', linewidth=2, color='#e67e22')
    ax.plot(sizes, results[op]['pytorch'], 's--', label='PyTorch', linewidth=2, color='#3498db')
    ax.set_xscale('log', base=2)
    ax.set_xlabel('Feature dimension (D)')
    ax.set_ylabel('Time (microseconds)')
    ax.set_title(title, fontsize=13)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f'JAX (XLA) vs PyTorch — {JAX_DEVICE} / {device}', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Speedup summary
print(f"{'Operation':<25s} {'D=256':>10s} {'D=1024':>10s} {'D=4096':>10s}")
print("-" * 60)
for op, title in zip(ops, titles):
    speedups = [j / p for j, p in zip(results[op]['pytorch'], results[op]['jax'])]
    # indices for D=256, 1024, 4096
    idx = [sizes.index(s) for s in [256, 1024, 4096]]
    vals = [f"{speedups[i]:.2f}x" for i in idx]
    print(f"{title:<25s} {vals[0]:>10s} {vals[1]:>10s} {vals[2]:>10s}")
print(f"\n(>1.0 = JAX faster, <1.0 = PyTorch faster)")

## 6. Where Each Approach Wins

| Scenario | Winner | Why |
|---|---|---|
| Standard ops (softmax, matmul) | **Tie** | Both call optimized libraries (cuBLAS, cuDNN) |
| Multi-op fusion (residual + norm) | **XLA** | Compiler auto-fuses; Triton needs a custom kernel per combination |
| Custom memory patterns (FlashAttention) | **Triton** | Online softmax + tiled matmul can't be expressed as standard op composition |
| Novel operations | **Triton** | If the compiler doesn't know the pattern, it can't optimize it |
| TPU deployment | **XLA** | Triton doesn't support TPU; XLA compiles the same code for both |
| Rapid prototyping | **XLA** | `@jax.jit` is one line; Triton kernel is 50+ lines |
| Maximum performance (known patterns) | **Triton** | Human can exploit hardware-specific tricks the compiler misses |

### The Key Insight

XLA and Triton operate at **different levels of abstraction**:

```
High level:  jax.jit(softmax)  →  XLA compiler  →  fused GPU code
Low level:   @triton.jit       →  you write it   →  GPU code
```

XLA is like an optimizing compiler (GCC -O3). Triton is like hand-written assembly. The compiler handles 90% of cases well. For the remaining 10% (FlashAttention, custom quantization, novel activations), you write kernels.

## 7. FlashAttention — Where XLA Can't Compete

Our [Fused Attention kernel](../06_Kernels/05_Fused_Attention.ipynb) uses **online softmax** — maintaining running max/sum state across K/V tiles. This isn't a composition of standard ops that XLA can fuse. It's a fundamentally different algorithm that requires manual kernel writing.

```python
# XLA can fuse this (standard ops):
@jax.jit
def attention_naive(Q, K, V):
    scores = Q @ K.T / jnp.sqrt(d)  # materializes N x N
    weights = jax.nn.softmax(scores)
    return weights @ V

# XLA CANNOT do this (needs custom algorithm):
# - Loop over K/V blocks with running softmax state
# - Rescale previous accumulations when max changes
# - Never materialize the N x N matrix
# → This requires Triton (or Pallas for JAX on TPU)
```

This is why Google developed **Pallas** — JAX's kernel writing framework — for exactly these cases where XLA isn't enough.

In [ ]:
# Demonstrate: JAX naive attention still materializes N x N
@jax.jit
def jax_naive_attention(Q, K, V):
    d = Q.shape[-1]
    scores = Q @ K.T / jnp.sqrt(d)     # N x N — XLA can't avoid this
    weights = jax.nn.softmax(scores, axis=-1)
    return weights @ V

# XLA will fuse the softmax part, but the N x N matrix still exists
N, d = 512, 64
Q = jnp.array(np.random.randn(N, d).astype(np.float32))
K = jnp.array(np.random.randn(N, d).astype(np.float32))
V = jnp.array(np.random.randn(N, d).astype(np.float32))

out = jax_naive_attention(Q, K, V)
print(f"JAX naive attention output: {out.shape}")
print(f"Score matrix size: {N}x{N} = {N*N*4/1024:.0f} KB")
print(f"\nXLA fuses the softmax ops but CANNOT avoid the N x N matrix.")
print(f"FlashAttention's online softmax trick requires a custom kernel.")

## What to Notice

1. **XLA auto-fuses standard op sequences.** Writing `mean → variance → normalize → scale` in JAX and `@jax.jit` often produces a single fused kernel — the same optimization our Triton LayerNorm kernel achieves manually.

2. **For standard ops, the difference is small.** Both frameworks call optimized libraries for matmul, and both handle softmax efficiently. The wins are at the margins.

3. **XLA wins at multi-op fusion.** Residual + LayerNorm, or any chain of element-wise + reduction ops, gets fused automatically. In Triton, you'd write a new kernel for each combination.

4. **Triton wins when the algorithm itself is non-standard.** FlashAttention's online softmax, custom quantization schemes, or novel attention patterns can't be expressed as op compositions. You need to write the kernel.

5. **XLA is portable, Triton is not.** The same `@jax.jit` code runs on GPU, TPU, and CPU. Triton only targets NVIDIA GPUs. For Google's TPU-centric infrastructure, this portability is essential.

6. **The 90/10 rule applies.** XLA handles 90% of optimization automatically. Triton exists for the 10% where the compiler isn't enough — and that 10% often matters most (attention is the bottleneck in LLMs).

## Resources

- [XLA Architecture](https://openxla.org/xla/architecture) — How XLA compiles and optimizes
- [JAX HLO Visualization](https://jax.readthedocs.io/en/latest/aot.html) — Inspect what XLA actually compiles
- [Pallas](https://jax.readthedocs.io/en/latest/pallas/index.html) — JAX's answer to Triton (custom kernels for TPU + GPU)
- [Triton Kernel Notebooks](../06_Kernels/) — Our hand-written kernels for comparison
- [torch.compile](https://pytorch.org/docs/stable/torch.compiler.html) — PyTorch's compiler (TorchInductor), a middle ground